****ETL****

**Importando as bibliotecas**

In [ ]:
import pandas as pd 
import os

**Trazendo os CSV's extraídos e criando os DataFrames**

In [ ]:
df_matches = pd.read_csv('../data/raw/WorldCupMatches.csv')
df_players = pd.read_csv('../data/raw/WorldCupPlayers.csv')
df_editions = pd.read_csv('../data/raw/WorldCups.csv')

**1) Analisando o DataFrame das partidas**

In [ ]:
df_matches.head()

,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930.0,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930.0,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930.0,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA
3,1930.0,14 Jul 1930 - 14:50,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,,2549.0,1.0,0.0,WARNKEN Alberto (CHI),LANGENUS Jean (BEL),MATEUCCI Francisco (URU),201.0,1098.0,ROU,PER
4,1930.0,15 Jul 1930 - 16:00,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,,23409.0,0.0,0.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),RADULESCU Constantin (ROU),201.0,1085.0,ARG,FRA


*Excluindo as colunas sem utilidade*

In [ ]:
colunas_excluidas_matches = [
    'Referee',
    'Assistant 1',
    'Assistant 2',
    'Home Team Initials',
    'Away Team Initials'
]

df_matches = df_matches.drop(columns=colunas_excluidas_matches)

*Criando a coluna de vencedores*

In [ ]:
def descobrir_vencedor(linha):
    gols_casa = linha['Home Team Goals']
    gols_fora = linha['Away Team Goals']
    time_casa = linha['Home Team Name']
    time_fora = linha['Away Team Name']
    
    condicao_vitoria = str(linha['Win conditions'])

    if gols_casa > gols_fora:
        return time_casa
    elif gols_fora > gols_casa:
        return time_fora
    else:
        if time_casa in condicao_vitoria:
            return time_casa
        elif time_fora in condicao_vitoria:
            return time_fora
        else:
            return 'Draw'

df_matches = df_matches.dropna(subset=['Home Team Name'])

df_matches['Winner'] = df_matches.apply(descobrir_vencedor, axis=1)

**2) Analisando o DataFrame de Jogadores**

In [ ]:
df_players.head()

,RoundID,MatchID,Team Initials,Coach Name,Line-up,Shirt Number,Player Name,Position,Event
0,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Alex THEPOT,GK,NaN
1,201,1096,MEX,LUQUE Juan (MEX),S,0,Oscar BONFIGLIO,GK,NaN
2,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Marcel LANGILLER,NaN,G40'
3,201,1096,MEX,LUQUE Juan (MEX),S,0,Juan CARRENO,NaN,G70'
4,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Ernest LIBERATI,NaN,NaN


*Excluindo Colunas sem utilidade*

In [ ]:
colunas_excluidas_players = [
   'Shirt Number', 
   'Line-up', 
   'Position'
]

df_players = df_players.drop(columns=colunas_excluidas_players, errors='ignore')

*Trocando inicial pelo nome do país* 

In [ ]:
df_players['Team Initials'].unique()

<ArrowStringArray>
['FRA', 'MEX', 'USA', 'BEL', 'YUG', 'BRA', 'ROU', 'PER', 'ARG', 'CHI', 'BOL',
 'PAR', 'URU', 'AUT', 'HUN', 'EGY', 'SUI', 'NED', 'SWE', 'GER', 'ESP', 'ITA',
 'TCH', 'INH', 'CUB', 'NOR', 'POL', 'ENG', 'SCO', 'FRG', 'TUR', 'KOR', 'URS',
 'WAL', 'NIR', 'COL', 'BUL', 'PRK', 'POR', 'ISR', 'MAR', 'SLV', 'GDR', 'AUS',
 'ZAI', 'HAI', 'TUN', 'IRN', 'CMR', 'NZL', 'ALG', 'HON', 'KUW', 'CAN', 'IRQ',
 'DEN', 'UAE', 'CRC', 'IRL', 'KSA', 'RUS', 'GRE', 'NGA', 'RSA', 'JPN', 'JAM',
 'CRO', 'SEN', 'SVN', 'ECU', 'CHN', 'TRI', 'CIV', 'SCG', 'ANG', 'CZE', 'GHA',
 'TOG', 'UKR', 'SRB', 'SVK', 'BIH']
Length: 82, dtype: str

In [ ]:
mapeamento_selecoes = {
    'FRA': 'France', 'MEX': 'Mexico', 'USA': 'USA', 'BEL': 'Belgium', 
    'YUG': 'Yugoslavia', 'BRA': 'Brazil', 'ROU': 'Romania', 'PER': 'Peru', 
    'ARG': 'Argentina', 'CHI': 'Chile', 'BOL': 'Bolivia', 'PAR': 'Paraguay', 
    'URU': 'Uruguay', 'AUT': 'Austria', 'HUN': 'Hungary', 'EGY': 'Egypt', 
    'SUI': 'Switzerland', 'NED': 'Netherlands', 'SWE': 'Sweden', 'GER': 'Germany', 
    'ESP': 'Spain', 'ITA': 'Italy', 'TCH': 'Czechoslovakia', 'INH': 'Dutch East Indies', 
    'CUB': 'Cuba', 'NOR': 'Norway', 'POL': 'Poland', 'ENG': 'England', 
    'SCO': 'Scotland', 'FRG': 'Germany FR', 'TUR': 'Turkey', 'KOR': 'Korea Republic', 
    'URS': 'Soviet Union', 'WAL': 'Wales', 'NIR': 'Northern Ireland', 'COL': 'Colombia', 
    'BUL': 'Bulgaria', 'PRK': 'Korea DPR', 'POR': 'Portugal', 'ISR': 'Israel', 
    'MAR': 'Morocco', 'SLV': 'El Salvador', 'GDR': 'German DR', 'AUS': 'Australia', 
    'ZAI': 'Zaire', 'HAI': 'Haiti', 'TUN': 'Tunisia', 'IRN': 'Iran', 
    'CMR': 'Cameroon', 'NZL': 'New Zealand', 'ALG': 'Algeria', 'HON': 'Honduras', 
    'KUW': 'Kuwait', 'CAN': 'Canada', 'IRQ': 'Iraq', 'DEN': 'Denmark', 
    'UAE': 'UAE', 'CRC': 'Costa Rica', 'IRL': 'Republic of Ireland', 'KSA': 'Saudi Arabia', 
    'RUS': 'Russia', 'GRE': 'Greece', 'NGA': 'Nigeria', 'RSA': 'South Africa', 
    'JPN': 'Japan', 'JAM': 'Jamaica', 'CRO': 'Croatia', 'SEN': 'Senegal', 
    'SVN': 'Slovenia', 'ECU': 'Ecuador', 'CHN': 'China PR', 'TRI': 'Trinidad and Tobago', 
    'CIV': "Côte d'Ivoire", 'SCG': 'Serbia and Montenegro', 'ANG': 'Angola', 'CZE': 'Czech Republic', 
    'GHA': 'Ghana', 'TOG': 'Togo', 'UKR': 'Ukraine', 'SRB': 'Serbia', 
    'SVK': 'Slovakia', 'BIH': 'Bosnia and Herzegovina'
}

df_players['Team Name'] = df_players['Team Initials'].map(mapeamento_selecoes)

df_players = df_players.drop(columns=['Team Initials'], errors='ignore')

**3) Analisando o DataFrame de Edições de copa**

In [ ]:
df_editions.head()

,Year,Country,Winner,Runners-Up,Third,Fourth,GoalsScored,QualifiedTeams,MatchesPlayed,Attendance
0,1930,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590.549
1,1934,Italy,Italy,Czechoslovakia,Germany,Austria,70,16,17,363.000
2,1938,France,Italy,Hungary,Brazil,Sweden,84,15,18,375.700
3,1950,Brazil,Uruguay,Brazil,Sweden,Spain,88,13,22,1.045.246
4,1954,Switzerland,Germany FR,Hungary,Austria,Uruguay,140,16,26,768.607


*Criando coluna de Média de Público*

In [ ]:
df_editions['Attendance'] = df_editions['Attendance'].astype(str).str.replace('.', '').astype(int)
df_editions['Avg Attendance'] = df_editions['Attendance'] / df_editions['MatchesPlayed']
df_editions['Avg Attendance'] = df_editions['Avg Attendance'].round(0).astype(int)

**Criando os DataFrames tratados**

In [ ]:
df_matches.to_parquet('../data/processed/matches_processed.parquet', index=False)
df_players.to_parquet('../data/processed/players_processed.parquet', index=False)
df_editions.to_parquet('../data/processed/editions_processed.parquet', index=False)